In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier

np.random.seed(42)

In [2]:
def generate_rct_treatment(row):
    return np.random.binomial(1, 0.5)


def generate_rct_selection(row):
    return 1


def generate_obs_treatment(row):
    if row["X1"] == -1:
        return np.random.binomial(1, 0.1)
    else:
        return np.random.binomial(1, 0.9)


def generate_outcome(row):
    if row["A"] == 0: 
        return row["X1"] + np.random.normal(0, 1)
    else: 
        if row["X1"] == -1:
            return 2 + row["X1"] + np.random.normal(0, 1)
        else:
            return 2 + row["X1"] + row["U1"] + np.random.normal(0, 1)


def generate_treatment_outcome_selection(df, fn_tr, fn_out, fn_sel):
    df["A"] = df.apply(fn_tr, axis=1)
    df["Y"] = df.apply(fn_out, axis=1)
    df["S"] = df.apply(fn_sel, axis=1) 


def generation_obs_selection(row):
    if row["X1"] == -1:
        return np.random.binomial(1, 0.1)
    else: 
        if row["U1"] == -1:
            return np.random.binomial(1, 0.9)
        else:
            return np.random.binomial(1, 0.1)
    

def log_om_res(df, covs, model): 

    if model == "DTR":
        om_A0 = DecisionTreeRegressor().fit(df.query("A==0")[covs], df.query("A==0")["Y"])
        om_A1 = DecisionTreeRegressor().fit(df.query("A==1")[covs], df.query("A==1")["Y"])

    df["hat_Y0"] = om_A0.predict(df[covs])  # log counter-factual outcome predictions for T=0
    df["hat_Y1"] = om_A1.predict(df[covs])  # log counter-factual outcome predictions for T=1

    df["SE_Y0"] = (df.query("A==0")["Y"] - df.query("A==0")["hat_Y0"]) ** 2  # log instance-wise squared-error of the outcome model for T=0
    df["SE_Y1"] = (df.query("A==1")["Y"] - df.query("A==1")["hat_Y1"]) ** 2  # log instance-wise squared-error of the outcome model for T=1
 

def log_ps_res(df, covs, model): 
    if model == "DTC":
        psm = DecisionTreeClassifier().fit(df[covs], df["A"])
        
    df["hat_P(A=1)"] = psm.predict_proba(df[covs])[:,1]
    df["BCELoss_A"] = - df["A"] * np.log(df["hat_P(A=1)"]) - (1 - df["A"]) * np.log(1 - df["hat_P(A=1)"]) 
    df["SE_A"] = (df["A"] - df["hat_P(A=1)"]) ** 2 


def log_rm_res(df, covs, model): 
    if model == "DTC":
        sm = DecisionTreeClassifier().fit(df[covs], df["R"])
    df["hat_P(R=1)"] = sm.predict_proba(df[covs])[:,1]
 

def log_sm_res(df, covs, model): 
    # probably don't need this model right now....
    if model == "DTC":
        sm = DecisionTreeClassifier().fit(df[covs], df["S"])
    df["hat_P(S=1)"] = sm.predict_proba(df[covs])[:,1]
    df["BCELoss_S"] = - df["S"] * np.log(df["hat_P(S=1)"]) - (1 - df["S"]) * np.log(1 - df["hat_P(S=1)"]) 
    df["SE_S"] = (df["S"] - df["hat_P(S=1)"]) ** 2 


def psi0(row):
    if row["R"] == 1:
        return 0
    else:
        t_ind = row["A"]
        p_r1, p_t1 = row["hat_P(R=1)"], row["hat_P(A=1)"]
        y, mu0, mu1 = row["Y"], row["hat_Y0"], row["hat_Y1"]

        term_1 = mu1 - mu0
        term_2 = t_ind * (y - mu1) / p_t1
        term_3 = (1 - t_ind) * (y - mu0) / (1 - p_t1)

        return (term_1 + term_2 - term_3)/ (1 - p_r1)


def psi1(row):
    if row["R"] == 0:
        return 0
    else:
        t_ind = row["A"]
        p_r1, p_t1 = row["hat_P(R=1)"], row["hat_P(A=1)"]
        y, mu0, mu1 = row["Y"], row["hat_Y0"], row["hat_Y1"]

        term_1 = mu1 - mu0
        term_2 = t_ind * (y - mu1) / p_t1
        term_3 = (1 - t_ind) * (y - mu0) / (1 - p_t1)

        return (term_1 + term_2 - term_3) / p_r1
    

def calc_contrasts(df):
    df["psi0"] = df.apply(psi0, axis=1)
    df["psi1"] = df.apply(psi1, axis=1)
    df["psi"] = df["psi1"] - df["psi0"]

In [3]:
n_covs = 1
n_unmeasured_covs = 1
covs = [f'X{i+1}' for i in range(n_covs)] 
unmeasured_covs = [f'U{i+1}' for i in range(n_unmeasured_covs)] 

n_rct = 1000
n_obs = 10000

X_rct = np.random.choice([-1, 1], size=(n_rct, n_covs))
U_rct = np.random.choice([-1, 1], size=(n_rct, n_unmeasured_covs))

df_rct = pd.DataFrame({**{cov: X_rct[:,i] for i, cov in enumerate(covs)},
                        **{u_cov: U_rct[:,i] for i, u_cov in enumerate(unmeasured_covs)},
                        **{'R': 1}})

X_obs = np.random.choice([-1, 1], size=(n_obs, n_covs))
U_obs = np.random.choice([-1, 1], size=(n_obs, n_unmeasured_covs))

df_obs = pd.DataFrame({**{cov: X_obs[:,i] for i, cov in enumerate(covs)},
                        **{u_cov: U_obs[:,i] for i, u_cov in enumerate(unmeasured_covs)},
                        **{'R': 0}})

In [4]:
obs_covs = covs 

generate_treatment_outcome_selection(df_rct, generate_rct_treatment, generate_outcome, generate_rct_selection)
log_om_res(df_rct, covs, "DTR")
log_ps_res(df_rct, covs, "DTC")
## no need for sm for RCT since everyone is selected

generate_treatment_outcome_selection(df_obs, generate_obs_treatment, generate_outcome, generation_obs_selection)
log_sm_res(df_obs, covs, "DTC")
df_obs = df_obs[df_obs["S"]==1].reset_index(drop=True) 
log_om_res(df_obs, covs, "DTR")
log_ps_res(df_obs, covs, "DTC")

df_merged = pd.concat([df_rct, df_obs])
log_rm_res(df_merged, covs, "DTC")


In [5]:
df_obs

,X1,U1,R,A,Y,S,hat_P(S=1),BCELoss_S,SE_S,hat_Y0,hat_Y1,SE_Y0,SE_Y1,hat_P(A=1),BCELoss_A,SE_A
0,1,-1,0,1,0.689385,1,0.509185,0.674943,0.240899,0.997239,2.187167,NaN,2.243351,0.896863,0.108852,0.010637
1,1,-1,0,1,0.093453,1,0.509185,0.674943,0.240899,0.997239,2.187167,NaN,4.383641,0.896863,0.108852,0.010637
2,1,-1,0,1,2.016447,1,0.509185,0.674943,0.240899,0.997239,2.187167,NaN,0.029145,0.896863,0.108852,0.010637
3,1,-1,0,1,1.202481,1,0.509185,0.674943,0.240899,0.997239,2.187167,NaN,0.969607,0.896863,0.108852,0.010637
4,-1,-1,0,0,0.012472,1,0.105569,2.248391,0.800007,-0.969894,0.983265,0.965043,NaN,0.098672,0.103886,0.009736
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3072,1,1,0,0,-0.288394,1,0.509185,0.674943,0.240899,0.997239,2.187167,1.652851,NaN,0.896863,2.271695,0.804363
3073,1,-1,0,1,2.350915,1,0.509185,0.674943,0.240899,0.997239,2.187167,NaN,0.026813,0.896863,0.108852,0.010637
3074,1,-1,0,1,2.159340,1,0.509185,0.674943,0.240899,0.997239,2.187167,NaN,0.000774,0.896863,0.108852,0.010637
3075,1,-1,0,0,0.024431,1,0.509185,0.674943,0.240899,0.997239,2.187167,0.946356,NaN,0.896863,2.271695,0.804363


In [6]:
calc_contrasts(df_merged)

In [7]:
df_merged["SE_A"].isna().sum()

0

In [8]:
# estimate covariance => we will computer both empirical estimate and SE

def compute_term2_helper(x1, x2, average=True): 
    cross_products = x1[:,None] * x2[None,:]
    mask = ~np.eye(len(x1), dtype=bool)
    if average: 
        return cross_products[mask].mean()
    else: 
        return cross_products[mask]


def compute_covariance_SE_outcome(df): 
    N = df.shape[0]
    trt_df = df.query("A==1")
    control_df = df.query("A==0")
    
    term1 = ((control_df["SE_Y0"]*np.abs(control_df["psi"])).sum() + (trt_df["SE_Y1"]*np.abs(trt_df["psi"])).sum())
    term1 = term1 / N
    
    # term 2 
    c1 = control_df["SE_Y0"].values; c2 = np.abs(control_df["psi"]).values 
    t1 = trt_df["SE_Y1"].values; t2 = np.abs(trt_df["psi"]).values 

    # Compute cross products for control group
    c_cross = compute_term2_helper(c1, c2, average=False)
    # Compute cross products for treatment group
    t_cross = compute_term2_helper(t1, t2, average=False)
    # Combine and compute mean
    term2 = np.mean(np.concatenate([c_cross, t_cross]))

    # covariance 
    return (N / (N-1)) * (term1 - term2) 


def compute_covariance_SE_treatment(df): 
    N = df.shape[0] 
    term1 = (df["SE_A"]*np.abs(df["psi"])).mean()
    # term 2 
    x1 = df["SE_A"].values; x2 = np.abs(df["psi"]).values 
    term2 = compute_term2_helper(x1, x2, average=True)
    # covariance 
    return (N / (N-1)) * (term1 - term2) 


def compute_covariance_SE_selection(df): 
    N = df.shape[0]
    term1 = (df["SE_S"]*np.abs(df["psi"])).mean()
    x1 = df["SE_S"].values; x2 = np.abs(df["psi"]).values 
    term2 = compute_term2_helper(x1, x2, average=True)
    return (N / (N-1)) * (term1 - term2) 
 

def compute_covariance_SE(df, signal="outcome"): 
    if signal == "outcome": 
        return compute_covariance_SE_outcome(df)
    elif signal == "treatment": 
        return compute_covariance_SE_treatment(df)
    elif signal == "selection": 
        return compute_covariance_SE_selection(df)
    else: 
        raise ValueError(f"covariance signal must be one of 'outcome', 'treatment', 'selection'")


In [10]:
df_obs = df_merged.query("R==0")
cov_Y = compute_covariance_SE(df_obs, signal="outcome")
print(cov_Y)
cov_A = compute_covariance_SE(df_obs, signal="treatment")
print(cov_A)
cov_S = compute_covariance_SE(df_obs, signal="selection")
print(cov_S)



3.027891991045112
0.6643531322563753
0.2252725487055241


In [11]:
# compute each signal separately for X1 = 1 and X1 = -1
# X1 = 1
df_x1_pos = df_obs.query("X1 == 1")
cov_Y_pos = compute_covariance_SE(df_x1_pos, signal="outcome")
cov_A_pos = compute_covariance_SE(df_x1_pos, signal="treatment") 
cov_S_pos = compute_covariance_SE(df_x1_pos, signal="selection")
print("X1 = 1:")
print(f"Outcome covariance: {cov_Y_pos}")
print(f"Treatment covariance: {cov_A_pos}")
print(f"Selection covariance: {cov_S_pos}")

# X1 = -1
df_x1_neg = df_obs.query("X1 == -1")
cov_Y_neg = compute_covariance_SE(df_x1_neg, signal="outcome")
cov_A_neg = compute_covariance_SE(df_x1_neg, signal="treatment")
cov_S_neg = compute_covariance_SE(df_x1_neg, signal="selection")
print("\nX1 = -1:")
print(f"Outcome covariance: {cov_Y_neg}")
print(f"Treatment covariance: {cov_A_neg}")
print(f"Selection covariance: {cov_S_neg}")


X1 = 1:
Outcome covariance: 3.126284489682496
Treatment covariance: 0.5763712099481675
Selection covariance: 0.0

X1 = -1:
Outcome covariance: 5.154730944086591
Treatment covariance: 1.1022971856339698
Selection covariance: -8.898669718288328e-16
